# Assignment 2 — Constrained SQL Generation

Qwen2.5-Coder-1.5B-Instruct generates SQLite queries for 12 natural-language questions, twice:

* **unconstrained**: ordinary sampling, no logits processor;
* **constrained**: the same call, with a Lark grammar enforced during decoding by Outlines.

Every output is checked against SQLite with `EXPLAIN`, and the constrained answers to the questions the grammar
cannot express are executed against `a2.db` to see what they actually return.

The notebook runs top to bottom in a fresh runtime. Grammar, prompts, seeds and settings are inline, and every
number in the tables is computed from this run.

## 0. Setup — library versions and the schema

In [1]:
# Colab already ships torch, pandas and sqlite3; these are the libraries whose APIs move.
%pip install -q "outlines==1.3.3" "llguidance==1.9.1" "lark==1.3.1" "transformers==5.18.0" "accelerate==1.15.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib.metadata as md
import platform
import sqlite3

import torch

for pkg in ["outlines", "outlines_core", "llguidance", "lark", "transformers", "accelerate", "torch", "pandas"]:
    print(f"{pkg:<14}{md.version(pkg)}")
print(f"{'python':<14}{platform.python_version()}")
print(f"{'sqlite':<14}{sqlite3.sqlite_version}")

outlines      1.3.3
outlines_core 0.2.14
llguidance    1.9.1
lark          1.3.1
transformers  5.18.0
accelerate    1.15.0
torch         2.14.1+cpu
pandas        3.0.6
python        3.12.3
sqlite        3.45.1


`a2.db` is the posted database. If it is not next to the notebook (for example in a fresh Colab runtime), the cell
below rebuilds it with the posted `seed_a2.py` logic. The seed is fixed, so the rebuilt file has exactly the same rows.

In [3]:
import os
import random

DB_PATH = "a2.db"

SCHEMA = """
CREATE TABLE users  (id INTEGER, name TEXT, city TEXT, age INTEGER);
CREATE TABLE orders (id INTEGER, user_id INTEGER, total REAL, status TEXT);
""".strip()


def build_a2_db(path: str) -> None:
    """Identical to the posted seed_a2.py."""
    random.seed(20260922)
    conn = sqlite3.connect(path)
    c = conn.cursor()
    c.execute("CREATE TABLE users  (id INTEGER, name TEXT, city TEXT, age INTEGER)")
    c.execute("CREATE TABLE orders (id INTEGER, user_id INTEGER, total REAL, status TEXT)")
    names = ["Alice", "Brandon", "Chen", "Dara", "Elena", "Femi", "Grace", "Hassan", "Iris",
             "Jamal", "Kiara", "Luis", "Maya", "Noah", "Olga", "Priya", "Quinn", "Rosa",
             "Samir", "Tessa", "Umar", "Vera", "Wes", "Xiomara", "Yusuf"]
    cities = ["Raleigh", "Greenville", "Durham", "Charlotte", "Wilmington"]
    users = [(i + 1, n, random.choice(cities), random.randint(19, 64)) for i, n in enumerate(names)]
    c.executemany("INSERT INTO users VALUES (?,?,?,?)", users)
    statuses = ["shipped", "pending", "cancelled", "delivered"]
    orders, oid = [], 1
    for uid in range(1, len(names) + 1):
        for _ in range(random.randint(0, 4)):
            orders.append((oid, uid, round(random.uniform(8.0, 480.0), 2), random.choice(statuses)))
            oid += 1
    c.executemany("INSERT INTO orders VALUES (?,?,?,?)", orders)
    conn.commit()
    conn.close()


if not os.path.exists(DB_PATH):
    build_a2_db(DB_PATH)

# Read-only: nothing the model writes can modify the database.
db = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
for (sql,) in db.execute("SELECT sql FROM sqlite_master WHERE type = 'table'"):
    print(sql + ";")
print("rows:", {t: db.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("users", "orders")})

CREATE TABLE users  (id INTEGER, name TEXT, city TEXT, age INTEGER);
CREATE TABLE orders (id INTEGER, user_id INTEGER, total REAL, status TEXT);
rows: {'users': 25, 'orders': 50}


## Part A — The grammar

This is the skeleton with the TODOs filled in. The only addition is `start : query`. Lark and llguidance (the
backend Outlines uses for CFGs) both treat `start` as the entry rule, so the same string works in both.

In [4]:
GRAMMAR = r"""
start         : query

query         : "SELECT" columns "FROM" table where_clause?
columns       : "*" | NAME ("," NAME)*
table         : "users" | "orders"
NAME          : "id" | "name" | "city" | "age" | "user_id" | "total" | "status"

where_clause  : "WHERE" condition
condition     : and_condition ("OR" and_condition)*
and_condition : comparison ("AND" comparison)*
comparison    : NAME COMP_OP literal

COMP_OP       : "=" | "!=" | "<=" | ">=" | "<" | ">"

literal       : FLOAT | INT | STRING
FLOAT         : /[0-9]+\.[0-9]+/
INT           : /[0-9]+/
STRING        : /'[^']*'/

%import common.WS
%ignore WS
"""

**What each rule does**

* `start : query`: the entry point. It contains exactly one `SELECT` statement and nothing else, so no trailing
  `;`, no prose and no second statement.
* `query`, `columns`, `table`, `NAME`: kept from the skeleton. `NAME` is a terminal (upper case) whose only
  strings are the seven column names, so the lexer cannot produce any other identifier. `columns` is either `*`
  or a comma-separated list of those names. `table` allows only the two tables.
* `condition` / `and_condition`: a condition is one or more `and_condition`s joined by `OR`, and each
  `and_condition` is one or more comparisons joined by `AND`. The two levels give `AND` higher precedence than
  `OR`, which matches SQL, so `a OR b AND c` parses as `a OR (b AND c)`, the same tree SQLite builds.
  Parentheses are not allowed, so `(a OR b) AND c` cannot be written. That is fine for this subset.
* `comparison : NAME COMP_OP literal`: the left side is always a known column and the right side is always a
  literal. Column-to-column comparisons are left out on purpose.
* `COMP_OP`: the six operators. It is a single terminal, and Lark's lexer takes the longest match, so `<=` lexes
  as one operator, not `<` followed by `=`.
* `literal` / `FLOAT` / `INT` / `STRING`: regular-expression terminals. `FLOAT` needs digits on both sides of
  the dot, so `100.` and `.5` are rejected. Longest match lexes `100.50` as one `FLOAT`, not `INT` `.` `INT`.
  `STRING` is a single-quoted run of non-quote characters, which is the SQL string syntax minus the `''` escape.
  Negative literals are left out because no column in this schema needs them.
* `%ignore WS`: whitespace (including newlines) may appear between any two terminals. Without it, a model token
  such as `" FROM"` (with a leading space) would not match the terminal `"FROM"`, and good output would be
  masked out.

### Four parse tests

In [5]:
from lark import Lark
from lark.exceptions import LarkError

parser = Lark(GRAMMAR, start="start")

PARSE_TESTS = [
    # (query, expected to parse?, why)
    ("SELECT name, age FROM users WHERE city = 'Raleigh' AND age < 30",
     True,  "column list, a string and an int literal, two comparisons joined by AND"),
    ("SELECT * FROM orders WHERE total >= 100.50 OR status != 'cancelled'",
     True,  "`*`, two-character operators, a FLOAT literal, and OR"),
    ("SELECT COUNT(*) FROM orders",
     False, "no aggregates: COUNT is not one of the seven allowed NAME strings, so the lexer stops at it"),
    ("SELECT name FROM users WHERE",
     False, "WHERE must be followed by a condition; the input ends early (UnexpectedEOF)"),
]

for query, expected, why in PARSE_TESTS:
    try:
        parser.parse(query)
        accepted, detail = True, ""
    except LarkError as err:
        accepted, detail = False, f"  [{type(err).__name__}]"
    assert accepted == expected, query
    print(f"{'ACCEPT' if accepted else 'REJECT'}  {query}\n        why: {why}{detail}\n")

print(parser.parse(PARSE_TESTS[0][0]).pretty())

ACCEPT  SELECT name, age FROM users WHERE city = 'Raleigh' AND age < 30
        why: column list, a string and an int literal, two comparisons joined by AND

ACCEPT  SELECT * FROM orders WHERE total >= 100.50 OR status != 'cancelled'
        why: `*`, two-character operators, a FLOAT literal, and OR

REJECT  SELECT COUNT(*) FROM orders
        why: no aggregates: COUNT is not one of the seven allowed NAME strings, so the lexer stops at it  [UnexpectedCharacters]

REJECT  SELECT name FROM users WHERE
        why: WHERE must be followed by a condition; the input ends early (UnexpectedEOF)  [UnexpectedEOF]

start
  query
    columns
      name
      age
    table
    where_clause
      condition
        and_condition
          comparison
            city
            =
            literal	'Raleigh'
          comparison
            age
            <
            literal	30



## Part B — Generate both ways

Both paths use **one** model object and **one** Outlines `Generator.__call__`, with identical prompt text, seeds
and sampling arguments. The only difference is the `output_type`:

* `outlines.Generator(model)` has `logits_processor = None`, so this is plain Hugging Face sampling.
* `outlines.Generator(model, CFG(GRAMMAR))` compiles the grammar into an llguidance logits processor. Hugging
  Face `generate` calls it at every decoding step, before sampling. It sets every token that cannot continue a
  grammatical prefix to −∞, and it allows end-of-sequence only once the prefix is a complete `query`. An illegal
  token never has nonzero probability, so this is not generate-then-filter.

In [6]:
import outlines
import transformers
from outlines.types import CFG

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

if torch.cuda.is_available():
    device, dtype = "cuda", torch.bfloat16
elif torch.backends.mps.is_available():
    device, dtype = "mps", torch.float16
else:
    # Not bfloat16: multithreaded bf16 CPU kernels gave different logits for identical inputs, which made
    # seeded sampling irreproducible. float32 was bit-identical across runs.
    device, dtype = "cpu", torch.float32
print("running on", device, dtype)

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
hf_model = transformers.AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
hf_model.eval()
model = outlines.from_transformers(hf_model, tokenizer)

unconstrained = outlines.Generator(model)
constrained = outlines.Generator(model, CFG(GRAMMAR))
print("unconstrained logits processor:", unconstrained.logits_processor)
print("constrained   logits processor:", type(constrained.logits_processor).__name__)

running on cpu torch.float32


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unconstrained logits processor: None
constrained   logits processor: LLGuidanceLogitsProcessor


**Settings**

* Temperature **0.8**, `do_sample=True`.
* Qwen's `generation_config.json` also sets `top_p=0.8`, `top_k=20` and `repetition_penalty=1.1`. They are
  overridden to neutral values (`top_p=1.0`, `top_k=0`, `repetition_penalty=1.0`), so the only sampling
  setting is temperature 0.8 and both conditions sample from the same distribution before masking.
* `max_new_tokens=128`, which is far more than any query in this subset needs.
* Seeds **`[1234, 5678]`**: sample *k* of every prompt uses `SEEDS[k]` in both conditions, via
  `transformers.set_seed` immediately before the call.
* Prompt: the schema and an instruction go in the system message, and the question in the user message. Both are
  rendered with Qwen's chat template. The rendered text is identical in both conditions.

**Reproducibility.** The saved outputs come from a CPU run in float32, which re-ran bit-for-bit identically.
Multithreaded bfloat16 on CPU did not: it gave different logits for identical inputs. On a GPU the seeds fix
the random stream, but different kernels can still move individual samples, so a Colab run may differ in a
few queries from the saved one. The tables are always computed from whatever run produced them, and the
discussion refers to the outputs saved in this notebook.

In [7]:
TEMPERATURE = 0.8
SEEDS = [1234, 5678]
MAX_NEW_TOKENS = 128
SAMPLING_KWARGS = dict(
    do_sample=True,
    temperature=TEMPERATURE,
    top_p=1.0,
    top_k=0,
    repetition_penalty=1.0,
    max_new_tokens=MAX_NEW_TOKENS,
)

SYSTEM_PROMPT = f"""You translate questions into SQLite queries for this database:

{SCHEMA}

Reply with a single SQL query and nothing else."""


def build_prompt(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


print(build_prompt("<question goes here>"))

<|im_start|>system
You translate questions into SQLite queries for this database:

CREATE TABLE users  (id INTEGER, name TEXT, city TEXT, age INTEGER);
CREATE TABLE orders (id INTEGER, user_id INTEGER, total REAL, status TEXT);

Reply with a single SQL query and nothing else.<|im_end|>
<|im_start|>user
<question goes here><|im_end|>
<|im_start|>assistant



### The 12 prompts

**Group 1** can be answered by the grammar: one table, optional column list, comparisons joined by AND/OR.
**Group 2** cannot be: each needs a join, an aggregate, `GROUP BY`, or `ORDER BY`/`LIMIT`.

Each prompt has a hand-written reference query. They are not used for generation. They are used only after
generation, to check whether a valid query also *answers the question* by returning the same rows. The Group 2
references deliberately go outside the grammar.

Some questions can reasonably be answered with different columns ("*who* lives in Raleigh" could return `name`
or `*`). For those, `alternatives` lists other projections that also count as correct. These alternatives were
added after looking at a first run, because the strict single-reference check was marking reasonable column
choices as wrong. They only widen the set of accepted *columns*. No alternative changes which rows count as the
answer.

In [8]:
from typing import NamedTuple


class Prompt(NamedTuple):
    group: int
    question: str
    reference_sql: str
    ordered: bool = False  # True when the question asks for a specific row order
    alternatives: tuple[str, ...] = ()  # other column choices that also answer the question


PROMPTS = [
    # Group 1 — expressible in the grammar
    Prompt(1, "Who are all the users that live in Raleigh?",
           "SELECT * FROM users WHERE city = 'Raleigh'",
           alternatives=("SELECT name FROM users WHERE city = 'Raleigh'",
                         "SELECT id, name FROM users WHERE city = 'Raleigh'")),
    Prompt(1, "Which orders have a total over 100 dollars?",
           "SELECT * FROM orders WHERE total > 100"),
    Prompt(1, "What are the names of users under 30?",
           "SELECT name FROM users WHERE age < 30"),
    Prompt(1, "List the id and total of every order that is still pending.",
           "SELECT id, total FROM orders WHERE status = 'pending'"),
    Prompt(1, "Show the names and ages of users who live in Durham and are older than 40.",
           "SELECT name, age FROM users WHERE city = 'Durham' AND age > 40"),
    Prompt(1, "Find all orders that were cancelled or cost less than 20 dollars.",
           "SELECT * FROM orders WHERE status = 'cancelled' OR total < 20"),
    Prompt(1, "Which city does Priya live in?",
           "SELECT city FROM users WHERE name = 'Priya'"),
    Prompt(1, "Show every order placed by the user with id 7.",
           "SELECT * FROM orders WHERE user_id = 7",
           alternatives=("SELECT id, total, status FROM orders WHERE user_id = 7",)),
    # Group 2 — needs a join, an aggregate, GROUP BY or ORDER BY
    Prompt(2, "How many orders did each user place?",
           "SELECT u.id, u.name, COUNT(o.id) FROM users u LEFT JOIN orders o ON o.user_id = u.id GROUP BY u.id"),
    Prompt(2, "What is the average order total?",
           "SELECT AVG(total) FROM orders"),
    Prompt(2, "What are the names of users who have placed an order over 300 dollars?",
           "SELECT DISTINCT u.name FROM users u JOIN orders o ON o.user_id = u.id WHERE o.total > 300"),
    Prompt(2, "List the five most expensive orders, from highest total to lowest.",
           "SELECT * FROM orders ORDER BY total DESC LIMIT 5", ordered=True,
           alternatives=("SELECT id, total FROM orders ORDER BY total DESC LIMIT 5",)),
]

assert sum(p.group == 1 for p in PROMPTS) == 8 and sum(p.group == 2 for p in PROMPTS) == 4
for group in (1, 2):
    print(f"Group {group}")
    for p in PROMPTS:
        if p.group == group:
            print(f"  {p.question}")

Group 1
  Who are all the users that live in Raleigh?
  Which orders have a total over 100 dollars?
  What are the names of users under 30?
  List the id and total of every order that is still pending.
  Show the names and ages of users who live in Durham and are older than 40.
  Find all orders that were cancelled or cost less than 20 dollars.
  Which city does Priya live in?
  Show every order placed by the user with id 7.
Group 2
  How many orders did each user place?
  What is the average order total?
  What are the names of users who have placed an order over 300 dollars?
  List the five most expensive orders, from highest total to lowest.


### Run all 48 generations (12 prompts × 2 conditions × 2 seeds)

In [9]:
import time

GENERATORS = {"Unconstrained": unconstrained, "Constrained": constrained}

records = []
start = time.perf_counter()
for prompt_idx, p in enumerate(PROMPTS):
    prompt_text = build_prompt(p.question)
    for sample_idx, seed in enumerate(SEEDS):
        for condition, generator in GENERATORS.items():
            transformers.set_seed(seed)
            output = generator(prompt_text, **SAMPLING_KWARGS)
            records.append(dict(condition=condition, group=p.group, prompt_idx=prompt_idx,
                                sample=sample_idx, seed=seed, question=p.question, output=output))
            print(f"[{condition:<13} g{p.group} q{prompt_idx:02d} seed={seed}] {output!r}")
print(f"\n{len(records)} generations in {time.perf_counter() - start:.0f}s")
assert len(records) == 48

[Unconstrained g1 q00 seed=1234] "SELECT name FROM users WHERE city = 'Raleigh'"


[Constrained   g1 q00 seed=1234] "SELECT name FROM users WHERE city = 'Raleigh'"


[Unconstrained g1 q00 seed=5678] "SELECT name FROM users WHERE city = 'Raleigh'"


[Constrained   g1 q00 seed=5678] "SELECT name FROM users WHERE city = 'Raleigh'"


[Unconstrained g1 q01 seed=1234] 'SELECT o.id, o.user_id, o.total, o.status FROM orders o WHERE o.total > 100'


...

[Constrained   g2 q10 seed=5678] 'SELECT user_id FROM orders WHERE total > 300'


[Unconstrained g2 q11 seed=1234] 'SELECT o.id, o.total FROM orders o ORDER BY o.total DESC LIMIT 5'


[Constrained   g2 q11 seed=1234] "SELECT * FROM orders WHERE status = 'active' AND total = 100000"


[Unconstrained g2 q11 seed=5678] 'SELECT T2.total FROM orders AS T1 INNER JOIN users AS T2 ON T1.user_id = T2.id ORDER BY T1.total DESC LIMIT 5'


[Constrained   g2 q11 seed=5678] "SELECT * FROM orders WHERE status != 'completed' AND total = 5"

48 generations in 151s


## Part C — Measure and look

**Validity** means SQLite accepts `EXPLAIN <query>` against `a2.db`. SQLite prepares the statement without
running it, so syntax is checked and table and column names are resolved against the schema. The grammar is
never used for this check.

The raw model output is what gets checked. An instruct model often wraps its answer in a Markdown code fence,
which is not SQL. For a fairer baseline there is a second column: the same check after the text inside the
first fence is extracted. The constrained path never needs this, because it cannot emit a backtick.

In [10]:
import re

import pandas as pd

FENCE = re.compile(r"```(?:sql|sqlite)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)


def strip_fence(text: str) -> str:
    match = FENCE.search(text)
    return match.group(1).strip() if match else text.strip()


def sqlite_error(query: str) -> str | None:
    """None if SQLite accepts the statement, else SQLite's error message."""
    try:
        db.execute(f"EXPLAIN {query}").fetchall()
        return None
    except (sqlite3.Error, sqlite3.Warning) as err:
        return f"{type(err).__name__}: {err}"


df = pd.DataFrame(records)
df["sql_stripped"] = df["output"].map(strip_fence)
df["error_raw"] = df["output"].map(sqlite_error)
df["error_stripped"] = df["sql_stripped"].map(sqlite_error)
df["valid_raw"] = df["error_raw"].isna()
df["valid_stripped"] = df["error_stripped"].isna()


def summarise(frame: pd.DataFrame) -> pd.DataFrame:
    order = {"Unconstrained": 0, "Constrained": 1}
    table = (
        frame.groupby(["condition", "group"])
        .agg(accepted=("valid_raw", "sum"), accepted_stripped=("valid_stripped", "sum"), n=("valid_raw", "size"))
        .reset_index()
        .sort_values(["condition", "group"], key=lambda s: s.map(order) if s.name == "condition" else s)
    )
    table["SQLite accepts"] = [f"{a}/{n} ({a / n:.0%})" for a, n in zip(table.accepted, table.n)]
    table["SQLite accepts (fence stripped)"] = [
        f"{a}/{n} ({a / n:.0%})" for a, n in zip(table.accepted_stripped, table.n)
    ]
    table = table.rename(columns={"condition": "Condition", "group": "Group"})
    return table[["Condition", "Group", "SQLite accepts", "SQLite accepts (fence stripped)", "n"]]


validity_table = summarise(df)
validity_table.style.hide(axis="index")

Condition,Group,SQLite accepts,SQLite accepts (fence stripped),n
Unconstrained,1,14/16 (88%),16/16 (100%),16
Unconstrained,2,7/8 (88%),7/8 (88%),8
Constrained,1,16/16 (100%),16/16 (100%),16
Constrained,2,6/8 (75%),6/8 (75%),8


Why SQLite rejected each query that failed:

In [11]:
pd.set_option("display.max_colwidth", None)
rejected = df[~df.valid_stripped]
print(rejected.groupby("condition").size().to_string(), "\n")
for row in rejected.itertuples():
    print(f"[{row.condition}, g{row.group}] {row.question}\n    {row.sql_stripped!r}\n    -> {row.error_stripped}\n")

condition
Constrained      2
Unconstrained    1 

[Constrained, g2] How many orders did each user place?
    'SELECT user_id, \n       name,\n       status,\n       total\nFROM orders'
    -> OperationalError: no such column: name

[Constrained, g2] How many orders did each user place?
    'SELECT user_id, name, city, age, status FROM users WHERE id = 1'
    -> OperationalError: no such column: user_id

[Unconstrained, g2] List the five most expensive orders, from highest total to lowest.
    'SELECT T2.total FROM orders AS T1 INNER JOIN users AS T2 ON T1.user_id = T2.id ORDER BY T1.total DESC LIMIT 5'
    -> OperationalError: no such column: T2.total



### Valid is not the same as correct

The grammar's guarantee stops at "SQLite would run this". To measure how much further a valid query gets, each
accepted query is executed, and its rows are compared with the reference query's rows. The comparison ignores
row order unless the question asks for one, and the selected columns have to match. By construction, no
constrained Group 2 query can match: the reference needs a construct the grammar does not have.

In [12]:
def run(query: str) -> list[tuple]:
    return db.execute(query).fetchall()


def matches_reference(row) -> bool:
    if not row.valid_stripped:
        return False
    prompt = PROMPTS[row.prompt_idx]
    got = run(row.sql_stripped)
    canonical = (lambda rows: rows) if prompt.ordered else (lambda rows: sorted(rows, key=repr))
    return any(canonical(got) == canonical(run(ref)) for ref in (prompt.reference_sql, *prompt.alternatives))


df["correct"] = df.apply(matches_reference, axis=1)

correctness = (
    df.groupby(["condition", "group"], sort=False)
    .agg(**{"SQLite accepts": ("valid_stripped", "sum"),
            "accepted and returns the reference rows": ("correct", "sum"),
            "n": ("correct", "size")})
    .reset_index()
    .rename(columns={"condition": "Condition", "group": "Group"})
)
correctness.style.hide(axis="index")

Condition,Group,SQLite accepts,accepted and returns the reference rows,n
Unconstrained,1,16,14,16
Constrained,1,16,14,16
Unconstrained,2,7,3,8
Constrained,2,6,0,8


In [13]:
print("Accepted by SQLite but wrong answer:\n")
for row in df[df.valid_stripped & ~df.correct].itertuples():
    print(f"[{row.condition}, g{row.group}] {row.question}\n    got: {row.sql_stripped}\n"
          f"    ref: {PROMPTS[row.prompt_idx].reference_sql}\n")

Accepted by SQLite but wrong answer:

[Unconstrained, g1] Find all orders that were cancelled or cost less than 20 dollars.
    got: SELECT * FROM orders WHERE status NOT IN ('completed', 'delivered') OR total < 20
    ref: SELECT * FROM orders WHERE status = 'cancelled' OR total < 20

[Constrained, g1] Find all orders that were cancelled or cost less than 20 dollars.
    got: SELECT * FROM orders WHERE status = 'Cancelled' OR total < 20
    ref: SELECT * FROM orders WHERE status = 'cancelled' OR total < 20

[Unconstrained, g1] Find all orders that were cancelled or cost less than 20 dollars.
    got: SELECT * FROM orders WHERE status != 'Confirmed' OR total < 20.0
    ref: SELECT * FROM orders WHERE status = 'cancelled' OR total < 20

[Constrained, g1] Find all orders that were cancelled or cost less than 20 dollars.
    got: SELECT * FROM orders WHERE status != 'Confirmed' OR total < 20.0
    ref: SELECT * FROM orders WHERE status = 'cancelled' OR total < 20

[Unconstrained, g2] How 

### The constrained Group 2 queries and what they returned

These four questions cannot be expressed in the grammar. The model could not refuse and could not leave the
grammar, so it wrote *something*. Each query is run against `a2.db`, and the result is printed next to the
question and the true answer.

In [14]:
PREVIEW_ROWS = 6


def show_rows(rows: list[tuple], indent: str = "      ") -> str:
    lines = [indent + repr(r) for r in rows[:PREVIEW_ROWS]]
    if len(rows) > PREVIEW_ROWS:
        lines.append(f"{indent}... ({len(rows) - PREVIEW_ROWS} more)")
    return "\n".join(lines) if lines else indent + "(no rows)"


for row in df[(df.condition == "Constrained") & (df.group == 2)].itertuples():
    reference = PROMPTS[row.prompt_idx].reference_sql
    print(f"Q: {row.question}   (seed {row.seed})")
    print(f"   constrained SQL: {row.output.strip()!r}")
    if row.valid_raw:
        rows = run(row.output)
        print(f"   returned {len(rows)} row(s):\n{show_rows(rows)}")
    else:
        print(f"   SQLite rejected it -> {row.error_raw}")
    truth = run(reference)
    print(f"   true answer ({reference}) -> {len(truth)} row(s):\n{show_rows(truth)}\n")

Q: How many orders did each user place?   (seed 1234)
   constrained SQL: 'SELECT user_id, \n       name,\n       status,\n       total\nFROM orders'
   SQLite rejected it -> OperationalError: no such column: name
   true answer (SELECT u.id, u.name, COUNT(o.id) FROM users u LEFT JOIN orders o ON o.user_id = u.id GROUP BY u.id) -> 25 row(s):
      (1, 'Alice', 4)
      (2, 'Brandon', 4)
      (3, 'Chen', 0)
      (4, 'Dara', 2)
      (5, 'Elena', 1)
      (6, 'Femi', 0)
      ... (19 more)

Q: How many orders did each user place?   (seed 5678)
   constrained SQL: 'SELECT user_id, name, city, age, status FROM users WHERE id = 1'
   SQLite rejected it -> OperationalError: no such column: user_id
   true answer (SELECT u.id, u.name, COUNT(o.id) FROM users u LEFT JOIN orders o ON o.user_id = u.id GROUP BY u.id) -> 25 row(s):
      (1, 'Alice', 4)
      (2, 'Brandon', 4)
      (3, 'Chen', 0)
      (4, 'Dara', 2)
      (5, 'Elena', 1)
      (6, 'Femi', 0)
      ... (19 more)

Q: What is the 

For comparison, the unconstrained Group 2 outputs:

In [15]:
for row in df[(df.condition == "Unconstrained") & (df.group == 2)].itertuples():
    status = "accepted" if row.valid_stripped else f"rejected ({row.error_stripped})"
    print(f"Q: {row.question}   (seed {row.seed})  [{status}, matches reference: {row.correct}]\n"
          f"   {row.sql_stripped!r}\n")

Q: How many orders did each user place?   (seed 1234)  [accepted, matches reference: False]
   'SELECT u.id, u.name, COUNT(o.id) as num_orders\nFROM users u\nJOIN orders o ON u.id = o.user_id\nGROUP BY u.id, u.name'

Q: How many orders did each user place?   (seed 5678)  [accepted, matches reference: False]
   'SELECT COUNT(*) ,  user_id FROM orders GROUP BY user_id'

Q: What is the average order total?   (seed 1234)  [accepted, matches reference: True]
   'SELECT AVG(total) FROM orders'

Q: What is the average order total?   (seed 5678)  [accepted, matches reference: True]
   'SELECT AVG(total) FROM orders'

Q: What are the names of users who have placed an order over 300 dollars?   (seed 1234)  [accepted, matches reference: False]
   'SELECT u.name FROM users u JOIN orders o ON u.id = o.user_id WHERE o.total > 300'

Q: What are the names of users who have placed an order over 300 dollars?   (seed 5678)  [accepted, matches reference: False]
   'SELECT T1.name FROM users AS T1 INNER JO

Where the constraint actually changed the output: with the same seed, the two conditions start from the same
random stream. Their outputs differ only where the unconstrained model wanted a token the grammar forbids (or
where renormalising over the allowed tokens moved a sample across a boundary).

In [16]:
paired = df.pivot_table(index=["group", "prompt_idx", "seed"], columns="condition", values="output", aggfunc="first")
paired["identical"] = paired["Constrained"].str.strip() == paired["Unconstrained"].str.strip()
print(paired.groupby("group")["identical"].agg(["sum", "size"]).rename(columns={"sum": "identical", "size": "pairs"}))
print("\nDistinct status values in a2.db:", [s for (s,) in db.execute("SELECT DISTINCT status FROM orders ORDER BY 1")])

       identical  pairs
group                  
1             11     16
2              0      8

Distinct status values in a2.db: ['cancelled', 'delivered', 'pending', 'shipped']


## Discussion

### 1. What did the constrained model produce for Group 2, and would a user notice?

The constrained model never refused, because the grammar has no way to say "I can't". Every Group 2 question got
a confident single-table `SELECT`. Taking the four questions in turn (the outputs are printed above):

* **"How many orders did each user place?"**: both samples began a column list that mixed the two tables
  (`SELECT user_id, name, status, total FROM orders`, and `SELECT user_id, name, city, age, status FROM users
  WHERE id = 1`). The model was clearly heading for a join. Every column name is a legal `NAME`, so the grammar
  let the list through, then forced a single table after `FROM`. SQLite rejected both (`no such column`). These
  are the **only two constrained queries SQLite rejected in the whole run**, so on Group 2 the constraint made
  validity *worse* than the unconstrained baseline (6/8 against 7/8). A user would notice, because they get an
  error.
* **"What is the average order total?"**: `SELECT total FROM orders` returned 50 raw totals instead of one
  number. The other sample added a filter on `status = 'completed'`, a status that does not exist, and returned
  zero rows. The first would probably be noticed (you asked for one number). The second looks like a legitimate
  answer: "no rows" reads as "no data", not "wrong query".
* **"Names of users who placed an order over 300 dollars"**: both samples gave
  `SELECT user_id FROM orders WHERE total > 300`. This is the most dangerous one. The filter is exactly right
  and the result looks plausible, but it is 21 rows of ids with repeats, not 15 names. Anyone counting rows
  would conclude 21 users qualify. Nothing about the output signals a problem.
* **"The five most expensive orders, highest first"**: with no `ORDER BY` or `LIMIT`, the model invented
  filters instead: `status = 'active' AND total = 100000` and `status != 'completed' AND total = 5`. Both
  are well-formed and schema-correct, and both return zero rows. A user sees an empty table and may conclude
  there are no orders.

So of the 8 constrained Group 2 queries, 6 pass the SQLite check and **0** return the right answer. The
unconstrained model, which was free to use `JOIN`, `AVG` and `ORDER BY`, answered 3 of 8 correctly. Its other
misses were subtle as well: an inner join that drops users with zero orders, and a join without `DISTINCT`
that repeats names. The constraint's only measurable gain in this run was formatting. It removed the two
Markdown-fenced answers in Group 1 (14/16 to 16/16 raw). Once those are stripped, the unconstrained model was
also accepted 16/16 on Group 1, so the constraint fixed form and did nothing for meaning.

### 2. The grammar hard-codes the column names. What does that rule out, and what if identifiers were arbitrary?

**Ruled out:** any identifier that does not exist in the schema. The model cannot emit a misspelled column
(`nmae`), a hallucinated one (`customer_name`, `order_date`), a different table (`customers`), an alias
(`num_orders`), or a qualified name (`o.total`, `T1.name`). The unconstrained run used aliases and
qualified names freely. Under the constraint, the same prompts came out as plain `SELECT * FROM orders WHERE
...`, because those tokens were masked. In SQLite's terms, the constraint eliminates **"no such table"**, and
**"no such column"** whenever the name exists nowhere in the schema.

**Not ruled out**, and our own results show it: a *real* column on the *wrong* table. `NAME` is one flat set
shared by both tables, and the column list comes *before* `FROM`, so nothing ties `name` to `users` or
`user_id` to `orders`. Both constrained rejections in this run are exactly that error. So is the one
unconstrained rejection (`T2.total`, where `T2` is `users`): hard-coding names did not touch the error class
that actually occurred.

**With arbitrary identifiers** (`NAME: /[a-z_]+/`), the grammar would accept every hallucinated name above. The
"SQLite accepts" rate for the constrained path would drop toward whatever the model's schema knowledge happens
to be, and the guarantee would shrink to pure syntax. It would also introduce a lexing problem: a lower-case
identifier regex matches `from`, `where`, `and` and `or`, so keywords would have to be excluded or prioritised
explicitly. Under BPE, the model could also build an identifier out of fragments that never appear in the
schema. Every additional bit of freedom in the grammar becomes another place the model can be wrong in a
well-formed way.

### 3. Two things about a SQL query that a context-free grammar cannot check

1. **Name resolution against the schema: does this column exist in *the table named in `FROM`*?** This is a
   declare-before-use (here, use-before-declare) constraint between two distant parts of the string. For our
   fixed two-table schema it could be faked by duplicating the query rule once per table, with per-table column
   terminals. That works only because the schema is tiny and baked into the grammar. For a general schema, where
   tables and columns are defined by `CREATE TABLE` statements, checking that every reference matches a
   declaration is not context-free, for the same reason a CFG cannot check that a program's variables are
   declared. SQLite's `EXPLAIN` does this check, and it is what caught our two constrained failures.
2. **Whether the query means what the user asked: the values and the intent.** The grammar knows `STRING` is
   `'...'`. It cannot know that the `status` column only holds `shipped`, `pending`, `cancelled` and `delivered`.
   In this run, the constrained model wrote `status = 'Cancelled'` (wrong case: SQLite's `=` is
   case-sensitive, so it matches nothing). Both conditions wrote `status != 'Confirmed'`, which keeps every
   row, and between them they also used the nonexistent statuses `'completed'` and `'active'`. All of them pass the grammar and SQLite, and all return the wrong rows. Likewise
   `SELECT user_id ...` for "names of users" and `SELECT total` for "the average" are grammatical, schema-correct
   and executable answers to different questions. This depends on the data and the user's intent, neither of
   which is in the string, so no grammar of any class over the query text can check it.

Type agreement is a third, smaller example: `age = 'thirty'` or `name > 5` parse and SQLite runs them under
dynamic typing. A grammar would need per-column literal rules to block them, and that is only possible because
the schema is finite and hard-coded.